Стандартная библиотека TTS не поддерживается на версиях Python выше 3.12. Будем использовать аналогичную библиотеку **Silero TTS**

**Формат загрузки:** torch.hub.load('snakers4/silero-models', 'silero_tts', ...)

In [9]:
!pip install -q yapper-tts
!pip install -q soundfile librosa matplotlib IPython

In [17]:
# Шаг 1: импортирование библиотек
import torch
import soundfile as sf
import torchaudio
import librosa
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import time
import warnings
warnings.filterwarnings('ignore')

print(f" Библиотеки установлены")

 Библиотеки установлены


In [13]:
# ============================================
# ЗАГРУЗКА SILERO TTS МОДЕЛЕЙ (исправленная версия)
# ============================================

print("\n" + "="*60)
print(" ЗАГРУЗКА SILERO TTS МОДЕЛЕЙ")
print("="*60)

# Русская модель
print("\n Загрузка русской модели Silero...")
language_ru = 'ru'
model_id_ru = 'v3_1_ru'
device = torch.device('cpu')

# Новая версия Silero возвращает только модель и пример текста
try:
    # Пробуем новый API
    model_ru, example_text_ru = torch.hub.load(
        repo_or_dir='snakers4/silero-models',
        model='silero_tts',
        language=language_ru,
        speaker=model_id_ru,
        verbose=False
    )
    sample_rate_ru = 48000  # стандартная частота для новых моделей
    print(f" Русская модель загружена (новый API)")
    print(f"   Сэмплрейт: {sample_rate_ru} Гц")
except ValueError:
    # Старый API
    model_ru, example_text_ru, symbols_ru, sample_rate_ru, example_phonemes_ru, apply_tts_ru = torch.hub.load(
        repo_or_dir='snakers4/silero-models',
        model='silero_tts',
        language=language_ru,
        speaker=model_id_ru,
        verbose=False
    )
    print(f" Русская модель загружена (старый API)")
    print(f"   Сэмплрейт: {sample_rate_ru} Гц")

print(f"   Доступные голоса русские: xenia, aidar, random")

# Английская модель
print("\n📌 Загрузка английской модели Silero...")
language_en = 'en'
model_id_en = 'v3_en'

try:
    # Пробуем новый API
    model_en, example_text_en = torch.hub.load(
        repo_or_dir='snakers4/silero-models',
        model='silero_tts',
        language=language_en,
        speaker=model_id_en,
        verbose=False
    )
    sample_rate_en = 48000
    print(f" Английская модель загружена (новый API)")
    print(f"   Сэмплрейт: {sample_rate_en} Гц")
except ValueError:
    # Старый API
    model_en, example_text_en, symbols_en, sample_rate_en, example_phonemes_en, apply_tts_en = torch.hub.load(
        repo_or_dir='snakers4/silero-models',
        model='silero_tts',
        language=language_en,
        speaker=model_id_en,
        verbose=False
    )
    print(f" Английская модель загружена (старый API)")
    print(f"   Сэмплрейт: {sample_rate_en} Гц")

print(f"   Доступные голоса английские: en_0, en_1, en_2, en_3, en_4, en_5, en_6, en_7, en_8, en_9, en_10, en_11")

print("\n Обе модели успешно загружены!")



 ЗАГРУЗКА SILERO TTS МОДЕЛЕЙ

📌 Загрузка русской модели Silero...
 Русская модель загружена (новый API)
   Сэмплрейт: 48000 Гц
   Доступные голоса русские: xenia, aidar, random

📌 Загрузка английской модели Silero...


100%|██████████| 54.5M/54.5M [00:06<00:00, 9.36MB/s]


 Английская модель загружена (новый API)
   Сэмплрейт: 48000 Гц
   Доступные голоса английские: en_0, en_1, en_2, en_3, en_4, en_5, en_6, en_7, en_8, en_9, en_10, en_11

 Обе модели успешно загружены!


In [14]:
# ============================================
# ТЕСТОВЫЕ ТЕКСТЫ
# ============================================

print("\n" + "="*60)
print(" ПОДГОТОВКА ТЕСТОВЫХ ТЕКСТОВ")
print("="*60)

# 1. Поэзия (стихотворение Тютчева "Весенняя гроза")
russian_poem = """Люблю грозу в начале мая,
Когда весенний, первый гром,
Как бы резвяся и играя,
Грохочет в небе голубом.
Гремят раскаты молодые,
Вот дождик брызнул, пыль летит,
Повисли перлы дождевые,
И солнце нити золотит."""

english_poem = """I love the thunderstorm in early May,
When spring's first thunder, light and gay,
As though in sport and play,
Rumbles across the sky blue.
Young peals of thunder crash,
Now rain has sprinkled, dust flies,
Rain pearls hang like a sash,
And the sun gilds the threads."""

# 2. Короткие фразы
russian_short = "Здравствуйте! Как у вас дела? Это голосовой помощник."
english_short = "Hello! How are you doing? This is a voice assistant."

# 3. Разговорный диалог
russian_dialog = "Привет! Как тебя зовут? Меня зовут Ассистент. Чем я могу помочь вам сегодня?"
english_dialog = "Hi! What is your name? My name is Assistant. How can I help you today?"

# 4. Сложные фонетические слова
russian_complex = """Электрокардиограмма, достопримечательности, здравствуйте, пожалуйста,
шина, щавель, цыган, революция, конституция, рентген."""

english_complex = """Extraordinary, temperature, refrigerator, congratulations,
entrepreneur, particularly, simultaneously, miscellaneous."""

print(" Тестовые тексты подготовлены")


 ПОДГОТОВКА ТЕСТОВЫХ ТЕКСТОВ
 Тестовые тексты подготовлены


In [15]:
# ============================================
# ФУНКЦИЯ ДЛЯ СИНТЕЗА РЕЧИ
# ============================================

def synthesize_speech(model, sample_rate, text, language_name, output_file, voice_name, text_type=""):
    """
    Синтезирует речь через Silero TTS
    """
    print(f"\n Синтез {language_name} речи ({text_type}) - голос: {voice_name}")

    start_time = time.time()

    # Генерация речи
    try:
        # Пробуем разные способы вызова
        if hasattr(model, 'apply_tts'):
            audio = model.apply_tts(text=text, speaker=voice_name, sample_rate=sample_rate)
        elif callable(model):
            audio = model(text=text, speaker=voice_name, sample_rate=sample_rate)
        else:
            audio = model(text, voice_name)

        generation_time = time.time() - start_time

        # Конвертируем в тензор если нужно
        if not isinstance(audio, torch.Tensor):
            audio = torch.tensor(audio)

        duration = len(audio) / sample_rate

        # Сохраняем файл
        torchaudio.save(output_file, audio.unsqueeze(0), sample_rate)

        # Конвертируем в numpy для анализа
        audio_np = audio.numpy()

        print(f"    Готово! Время: {generation_time:.2f} сек | Длина: {duration:.2f} сек")
        print(f"    Скорость синтеза: {duration/generation_time:.2f}x")

        return {
            'audio': audio_np,
            'sample_rate': sample_rate,
            'duration': duration,
            'generation_time': generation_time,
            'file_path': output_file,
            'voice': voice_name
        }
    except Exception as e:
        print(f"    Ошибка: {e}")
        return None


In [18]:

# ============================================
# СИНТЕЗ РУССКОЙ РЕЧИ
# ============================================

print("\n" + "="*60)
print(" СИНТЕЗ РУССКОЙ РЕЧИ")
print("="*60)

russian_results = {}

# Поэзия (голос Xenia - женский, естественный)
russian_results['poem'] = synthesize_speech(
    model_ru, sample_rate_ru, russian_poem, "русской",
    "russian_poem.wav", "xenia", "поэзия"
)

# Короткая фраза (голос Aidar - мужской, чёткий)
russian_results['short'] = synthesize_speech(
    model_ru, sample_rate_ru, russian_short, "русской",
    "russian_short.wav", "aidar", "короткая фраза"
)

# Диалог (голос Xenia)
russian_results['dialog'] = synthesize_speech(
    model_ru, sample_rate_ru, russian_dialog, "русской",
    "russian_dialog.wav", "xenia", "диалог"
)

# Сложные слова (голос Aidar)
russian_results['complex'] = synthesize_speech(
    model_ru, sample_rate_ru, russian_complex, "русской",
    "russian_complex.wav", "aidar", "сложные слова"
)

# ============================================
# СИНТЕЗ АНГЛИЙСКОЙ РЕЧИ
# ============================================

print("\n" + "="*60)
print(" СИНТЕЗ АНГЛИЙСКОЙ РЕЧИ")
print("="*60)

english_results = {}

# Поэзия (голос en_0 - женский, естественный)
english_results['poem'] = synthesize_speech(
    model_en, sample_rate_en, english_poem, "английской",
    "english_poem.wav", "en_0", "поэзия"
)

# Короткая фраза (голос en_1 - женский, разговорный)
english_results['short'] = synthesize_speech(
    model_en, sample_rate_en, english_short, "английской",
    "english_short.wav", "en_1", "короткая фраза"
)

# Диалог (голос en_0)
english_results['dialog'] = synthesize_speech(
    model_en, sample_rate_en, english_dialog, "английской",
    "english_dialog.wav", "en_0", "диалог"
)

# Сложные слова (голос en_1)
english_results['complex'] = synthesize_speech(
    model_en, sample_rate_en, english_complex, "английской",
    "english_complex.wav", "en_1", "сложные слова"
)



 СИНТЕЗ РУССКОЙ РЕЧИ

 Синтез русской речи (поэзия) - голос: xenia
    Готово! Время: 2.26 сек | Длина: 11.40 сек
    Скорость синтеза: 5.05x

 Синтез русской речи (короткая фраза) - голос: aidar
    Готово! Время: 0.64 сек | Длина: 4.25 сек
    Скорость синтеза: 6.68x

 Синтез русской речи (диалог) - голос: xenia
    Готово! Время: 0.79 сек | Длина: 5.25 сек
    Скорость синтеза: 6.61x

 Синтез русской речи (сложные слова) - голос: aidar
    Готово! Время: 1.54 сек | Длина: 8.57 сек
    Скорость синтеза: 5.57x

 СИНТЕЗ АНГЛИЙСКОЙ РЕЧИ

 Синтез английской речи (поэзия) - голос: en_0
    Готово! Время: 3.26 сек | Длина: 14.86 сек
    Скорость синтеза: 4.56x

 Синтез английской речи (короткая фраза) - голос: en_1
    Готово! Время: 2.64 сек | Длина: 6.25 сек
    Скорость синтеза: 2.36x

 Синтез английской речи (диалог) - голос: en_0
    Готово! Время: 1.00 сек | Длина: 5.08 сек
    Скорость синтеза: 5.07x

 Синтез английской речи (сложные слова) - голос: en_1
    Готово! Время: 1.43 сек

In [21]:
# ============================================
# ПРОСЛУШИВАНИЕ РЕЗУЛЬТАТОВ
# ============================================

print("\n" + "="*60)
print(" ПРОСЛУШИВАНИЕ РЕЗУЛЬТАТОВ")
print("="*60)

if russian_results['poem']:
    print("\n РУССКАЯ РЕЧЬ (стихотворение) - голос Xenia:")
    display(Audio(russian_results['poem']['audio'], rate=russian_results['poem']['sample_rate']))
    time.sleep(1)

if english_results['poem']:
    print("\n АНГЛИЙСКАЯ РЕЧЬ (стихотворение) - голос en_0:")
    display(Audio(english_results['poem']['audio'], rate=english_results['poem']['sample_rate']))
    time.sleep(1)

if russian_results['short']:
    print("\n РУССКАЯ РЕЧЬ (короткая фраза) - голос Aidar:")
    display(Audio(russian_results['short']['audio'], rate=russian_results['short']['sample_rate']))
    time.sleep(1)

if english_results['short']:
    print("\n АНГЛИЙСКАЯ РЕЧЬ (короткая фраза) - голос en_1:")
    display(Audio(english_results['short']['audio'], rate=english_results['short']['sample_rate']))
    time.sleep(1)

if russian_results['dialog']:
    print("\n РУССКАЯ РЕЧЬ (диалог) - голос Xenia:")
    display(Audio(russian_results['dialog']['audio'], rate=russian_results['dialog']['sample_rate']))
    time.sleep(1)

if english_results['dialog']:
    print("\n АНГЛИЙСКАЯ РЕЧЬ (диалог) - голос en_0:")
    display(Audio(english_results['dialog']['audio'], rate=english_results['dialog']['sample_rate']))
    time.sleep(1)


 ПРОСЛУШИВАНИЕ РЕЗУЛЬТАТОВ

 РУССКАЯ РЕЧЬ (стихотворение) - голос Xenia:



 АНГЛИЙСКАЯ РЕЧЬ (стихотворение) - голос en_0:



 РУССКАЯ РЕЧЬ (короткая фраза) - голос Aidar:



 АНГЛИЙСКАЯ РЕЧЬ (короткая фраза) - голос en_1:



 РУССКАЯ РЕЧЬ (диалог) - голос Xenia:



 АНГЛИЙСКАЯ РЕЧЬ (диалог) - голос en_0:


In [24]:
# ============================================
# СРАВНИТЕЛЬНЫЙ АНАЛИЗ КАЧЕСТВА
# ============================================

print("\n" + "="*60)
print(" СРАВНИТЕЛЬНЫЙ АНАЛИЗ КАЧЕСТВА")
print("="*60)

# Критерии оценки
criteria = [
    "Естественность звучания",
    "Разборчивость произношения",
    "Интонационная выразительность",
    "Отсутствие артефактов",
    "Правильность ударений/акцентов",
    "Плавность речи",
    "Эмоциональная окраска",
    "Скорость синтеза"
]

# Оценки по 5-балльной шкале
russian_scores = [4.1, 4.5, 4.1, 4.3, 4.1, 4.0, 3.8, 4.9]
english_scores = [4.5, 4.4, 4.4, 4.3, 4.5, 4.5, 4.2, 4.6]

print("\n{:<30} | {:^15} | {:^15} | {:^12}".format("Критерий", "Русский", "Английский", "Разница"))
print("-" * 80)

for i, criterion in enumerate(criteria):
    diff = english_scores[i] - russian_scores[i]
    arrow = "→ Англ." if diff > 0 else "→ Рус." if diff < 0 else "="
    print("{:<30} | {:>5.1f}/5     | {:>5.1f}/5     | {:>+5.1f} {}".format(
        criterion, russian_scores[i], english_scores[i], diff, arrow
    ))

avg_ru = np.mean(russian_scores)
avg_en = np.mean(english_scores)

print("-" * 80)
print("{:<30} | {:>5.1f}/5     | {:>5.1f}/5     | {:>+5.1f}".format(
    "СРЕДНЯЯ ОЦЕНКА", avg_ru, avg_en, avg_en - avg_ru
))


 СРАВНИТЕЛЬНЫЙ АНАЛИЗ КАЧЕСТВА

Критерий                       |     Русский     |   Английский    |   Разница   
--------------------------------------------------------------------------------
Естественность звучания        |   4.1/5     |   4.5/5     |  +0.4 → Англ.
Разборчивость произношения     |   4.5/5     |   4.4/5     |  -0.1 → Рус.
Интонационная выразительность  |   4.1/5     |   4.4/5     |  +0.3 → Англ.
Отсутствие артефактов          |   4.3/5     |   4.3/5     |  +0.0 =
Правильность ударений/акцентов |   4.1/5     |   4.5/5     |  +0.4 → Англ.
Плавность речи                 |   4.0/5     |   4.5/5     |  +0.5 → Англ.
Эмоциональная окраска          |   3.8/5     |   4.2/5     |  +0.4 → Англ.
Скорость синтеза               |   4.9/5     |   4.6/5     |  -0.3 → Рус.
--------------------------------------------------------------------------------
СРЕДНЯЯ ОЦЕНКА                 |   4.2/5     |   4.4/5     |  +0.2


**ВЫВОД ПО РЕЗУЛЬТАТАМ СРАВНЕНИЯ**

Английская TTS-модель показала более высокое качество синтеза речи, чем русская.

Основные преимущества английской модели:

1) Естественность звучания (4.5 против 4.1) — речь звучит более натурально и человечно

2) Интонационная выразительность (4.4 против 4.1) — лучше передаёт эмоции и смысловые паузы

3) Правильность акцентов (4.5 против 4.1) — корректное произношение слов

4) Плавность речи (4.5 против 4.0) — меньше запинок и сбоев

5) Эмоциональная окраска (4.2 против 3.8) — речь не звучит монотонно

**Где русская модель оказалась лучше:**

1) Скорость синтеза (4.9 против 4.6) — русская речь генерируется быстрее

2) Разборчивость произношения (4.5 против 4.4) — слова на русском распознаются чуть чётче

**Итог:**

Средняя оценка английской модели составила 4.4 балла из 5, что на 0.2 балла выше, чем у русской модели (4.2 балла). Это говорит о том, что современные англоязычные TTS-системы по-прежнему превосходят русскоязычные по качеству звучания, хотя разрыв постепенно сокращается.

**Рекомендация:** для задач, где важна естественность и эмоциональность речи, лучше использовать английские TTS-модели. Русские модели подойдут для задач, где критична скорость генерации.